# SAP RPT-1 via Native SDK (`RPTClient`)

This notebook demonstrates how to use the **SAP Generative AI Hub native SDK** to make predictions with SAP RPT-1.

The SDK provides a high-level `RPTClient` that handles authentication, deployment resolution, and request serialization internally — replacing the manual HTTP workflow with a few lines of code.

It covers:
- **Classification** predictions using Pydantic models and row-based data
- **Regression** predictions using plain dictionaries and column-based data
- **Mixed task types** (classification + regression) in a single call
- Response interpretation (predictions, confidence, metadata)
- **Pandas DataFrame** workflow — load a CSV, split context/query, predict

> RPT-1 is a **Relational Pretrained Transformer** for tabular/relational data. It solves classification and regression tasks out-of-the-box via in-context learning — no training or fine-tuning required.

## 1) Prerequisites

### Python packages
```bash
pip install -r requirements.txt
```

### Environment variables

The SDK reads AI Core credentials from environment variables. Set these in `.env`:

```bash
AICORE_AUTH_URL="<YOUR_AUTH_URL>"
AICORE_CLIENT_ID="<YOUR_CLIENT_ID>"
AICORE_CLIENT_SECRET="<YOUR_CLIENT_SECRET>"
AICORE_BASE_URL="<YOUR_AI_API_URL>"
AICORE_RESOURCE_GROUP="default"
```

### Deployment

An RPT-1 deployment must be running on SAP AI Core. You can create one through the [AI Launchpad](https://developers.sap.com/tutorials/ai-core-generative-ai.html#6c4a539e-2bdf-4ddb-97a0-0f8d0f1bd00e) or via the API (see the `RPT-1_request.ipynb` notebook).

The SDK resolves the deployment automatically — no deployment URL or ID is needed in the code.

## 2) Setup

Load the `.env` file **before** importing `gen_ai_hub` so the SDK picks up the credentials automatically.

In [8]:
from dotenv import load_dotenv

load_dotenv()

from gen_ai_hub.proxy.native.sap.client import RPTClient
from gen_ai_hub.proxy.native.sap.models import (
    RPTRequest,
    PredictionConfig,
    TargetColumn,
)

# There are two models available: "sap-rpt-1-small" and "sap-rpt-1-large"
MODEL_NAME = "sap-rpt-1-small"
client = RPTClient()

print("Client initialized.")

Client initialized.


## 3) Classification (Pydantic models + row-based data)

The SDK provides Pydantic models (`RPTRequest`, `PredictionConfig`, `TargetColumn`) for structured, type-safe request construction.

In **row-based** format, each record is a dictionary inside the `rows` list. Place `"[PREDICT]"` in the target column for rows you want predicted. Rows without the placeholder serve as context — the model uses them to learn patterns for its predictions.

In [9]:
classification_request = RPTRequest(
    prediction_config=PredictionConfig(
        target_columns=[
            TargetColumn(name="COSTCENTER", task_type="classification")
        ]
    ),
    rows=[
        {"PRODUCT": "Couch",        "PRICE": 999.99, "ORDERDATE": "28-11-2025", "ID": "35",  "COSTCENTER": "[PREDICT]"},
        {"PRODUCT": "Office Chair", "PRICE": 150.80, "ORDERDATE": "02-11-2025", "ID": "44",  "COSTCENTER": "Office Furniture"},
        {"PRODUCT": "Server Rack",  "PRICE": 2200.00,"ORDERDATE": "01-11-2025", "ID": "104", "COSTCENTER": "Data Infrastructure"},
        {"PRODUCT": "Standing Desk","PRICE": 640.00, "ORDERDATE": "05-11-2025", "ID": "205", "COSTCENTER": "Office Furniture"},
    ],
    index_column="ID",
)

response = client.predict(body=classification_request, model_name=MODEL_NAME)

print("Status:     ", response.status)
print("Metadata:   ", response.metadata)
print("Predictions:", response.predictions)

Status:      code=0 message='ok'
Metadata:    num_rows=4 num_columns=5 num_predictions=1 num_query_rows=1
Predictions: [Prediction(root={'COSTCENTER': [PredictionItem(prediction='Office Furniture', confidence=0.84)], 'ID': 35})]


## 4) Regression (plain dictionary + column-based data)

The `RPTClient.predict()` method also accepts a **plain dictionary** instead of a Pydantic model. This is useful when building payloads dynamically.

In **column-based** format, data is passed as arrays under a `columns` key. This example also includes an explicit `data_schema` — while optional, providing it is recommended for best prediction quality.

In [10]:
regression_request = {
    "prediction_config": {
        "target_columns": [
            {
                "name": "DISCOUNT_RATE",
                "task_type": "regression",
            }
        ]
    },
    "columns": {
        "PRODUCT":       ["Couch", "Office Chair", "Server Rack", "Standing Desk", "Monitor 27 inch"],
        "PRICE":         [999.99, 150.80, 2200.00, 640.00, 289.99],
        "ORDERDATE":     ["28-11-2025", "02-11-2025", "01-11-2025", "05-11-2025", "08-11-2025"],
        "ID":            ["35", "44", "104", "205", "306"],
        "DISCOUNT_RATE": ["[PREDICT]", 0.12, 0.05, 0.10, "[PREDICT]"],
    },
    "data_schema": {
        "PRODUCT":       {"dtype": "string"},
        "PRICE":         {"dtype": "numeric"},
        "ORDERDATE":     {"dtype": "date"},
        "ID":            {"dtype": "string"},
        "DISCOUNT_RATE": {"dtype": "numeric"},
    },
    "index_column": "ID",
}

response = client.predict(body=regression_request, model_name=MODEL_NAME)

print("Status:     ", response.status)
print("Metadata:   ", response.metadata)
print("Predictions:", response.predictions)

Status:      code=0 message='ok'
Metadata:    num_rows=5 num_columns=5 num_predictions=2 num_query_rows=2
Predictions: [Prediction(root={'DISCOUNT_RATE': [PredictionItem(prediction=0.09109286963939667, confidence=None)], 'ID': '35'}), Prediction(root={'DISCOUNT_RATE': [PredictionItem(prediction=0.09129524976015091, confidence=None)], 'ID': '306'})]


## 5) Mixed task types (classification + regression in one call)

RPT-1 supports predicting **multiple target columns** with different task types in a single request. Here we predict `COSTCENTER` (classification) and `DISCOUNT_RATE` (regression) simultaneously.

The `[PREDICT]` placeholder is placed independently per target column — a row can have a placeholder in one target and a known value in another.

In [11]:
mixed_request = RPTRequest(
    prediction_config=PredictionConfig(
        target_columns=[
            TargetColumn(name="COSTCENTER", task_type="classification"),
            TargetColumn(name="DISCOUNT_RATE", task_type="regression"),
        ]
    ),
    rows=[
        {"PRODUCT": "Couch",         "PRICE": 999.99,  "ORDERDATE": "28-11-2025", "ID": "35",
         "COSTCENTER": "[PREDICT]",  "DISCOUNT_RATE": "[PREDICT]"},
        {"PRODUCT": "Office Chair",  "PRICE": 150.80,  "ORDERDATE": "02-11-2025", "ID": "44",
         "COSTCENTER": "Office Furniture", "DISCOUNT_RATE": 0.12},
        {"PRODUCT": "Server Rack",   "PRICE": 2200.00, "ORDERDATE": "01-11-2025", "ID": "104",
         "COSTCENTER": "Data Infrastructure", "DISCOUNT_RATE": 0.05},
        {"PRODUCT": "Standing Desk", "PRICE": 640.00,  "ORDERDATE": "05-11-2025", "ID": "205",
         "COSTCENTER": "Office Furniture", "DISCOUNT_RATE": 0.10},
        {"PRODUCT": "Monitor 27 inch", "PRICE": 289.99, "ORDERDATE": "08-11-2025", "ID": "306",
         "COSTCENTER": "[PREDICT]",  "DISCOUNT_RATE": "[PREDICT]"},
    ],
    index_column="ID",
    data_schema={
        "PRODUCT":       {"dtype": "string"},
        "PRICE":         {"dtype": "numeric"},
        "ORDERDATE":     {"dtype": "date"},
        "ID":            {"dtype": "string"},
        "COSTCENTER":    {"dtype": "string"},
        "DISCOUNT_RATE": {"dtype": "numeric"},
    },
)

response = client.predict(body=mixed_request, model_name=MODEL_NAME)

print("Status:     ", response.status)
print("Metadata:   ", response.metadata)
print("Predictions:", response.predictions)

Status:      code=0 message='ok'
Metadata:    num_rows=5 num_columns=6 num_predictions=4 num_query_rows=2
Predictions: [Prediction(root={'COSTCENTER': [PredictionItem(prediction='Office Furniture', confidence=0.89)], 'DISCOUNT_RATE': [PredictionItem(prediction=0.09109286963939667, confidence=None)], 'ID': '35'}), Prediction(root={'COSTCENTER': [PredictionItem(prediction='Office Furniture', confidence=0.96)], 'DISCOUNT_RATE': [PredictionItem(prediction=0.09129524976015091, confidence=None)], 'ID': '306'})]


## 6) Interpreting the response

The response object has three key attributes:

| Attribute | Description |
|---|---|
| `status` | Contains `code` (0=OK, 1=warning, 2=invalid input, 3=server error) and `message`. |
| `metadata` | Row/column counts: `num_rows`, `num_columns`, `num_predictions`, `num_query_rows`. |
| `predictions` | A list of `Prediction` objects, one per query row. |

Each `Prediction` contains one or more target columns, each with a list of `PredictionItem` objects:

| Field | Classification | Regression |
|---|---|---|
| `prediction` | Predicted label (string) | Predicted value (float) |
| `confidence` | Probability score (0-1) | `None` |

Let's unpack the mixed-task response to see both types side by side.

In [12]:
for i, pred in enumerate(response.predictions):
    row_data = pred.root
    row_id = row_data.get("ID", f"row_{i}")
    print(f"\n--- Query Row (ID={row_id}) ---")

    for col_name, items in row_data.items():
        if not isinstance(items, list):
            continue
        for item in items:
            print(f"  {col_name}:")
            print(f"    prediction = {item.prediction}")
            if item.confidence is not None:
                print(f"    confidence = {item.confidence}")
            else:
                print(f"    confidence = N/A (regression)")


--- Query Row (ID=35) ---
  COSTCENTER:
    prediction = Office Furniture
    confidence = 0.89
  DISCOUNT_RATE:
    prediction = 0.09109286963939667
    confidence = N/A (regression)

--- Query Row (ID=306) ---
  COSTCENTER:
    prediction = Office Furniture
    confidence = 0.96
  DISCOUNT_RATE:
    prediction = 0.09129524976015091
    confidence = N/A (regression)


## 7) Classification with `top_k`

For classification tasks, you can set `top_k` to receive the top N predicted labels per cell instead of just the most likely one. This is useful when you want to see alternative predictions and their confidence scores.

In [13]:
topk_request = RPTRequest(
    prediction_config=PredictionConfig(
        target_columns=[
            TargetColumn(name="COSTCENTER", task_type="classification", top_k=3)
        ]
    ),
    rows=[
        {"PRODUCT": "Couch",        "PRICE": 999.99, "ORDERDATE": "28-11-2025", "ID": "35",  "COSTCENTER": "[PREDICT]"},
        {"PRODUCT": "Office Chair", "PRICE": 150.80, "ORDERDATE": "02-11-2025", "ID": "44",  "COSTCENTER": "Office Furniture"},
        {"PRODUCT": "Server Rack",  "PRICE": 2200.00,"ORDERDATE": "01-11-2025", "ID": "104", "COSTCENTER": "Data Infrastructure"},
        {"PRODUCT": "Standing Desk","PRICE": 640.00, "ORDERDATE": "05-11-2025", "ID": "205", "COSTCENTER": "Office Furniture"},
    ],
    index_column="ID",
)

response = client.predict(body=topk_request, model_name=MODEL_NAME)

for pred in response.predictions:
    row_data = pred.root
    row_id = row_data.get("ID", "?")
    print(f"\nID={row_id} — top {len(row_data['COSTCENTER'])} predictions:")
    for rank, item in enumerate(row_data["COSTCENTER"], start=1):
        print(f"  {rank}. {item.prediction} (confidence={item.confidence})")


ID=35 — top 1 predictions:
  1. Office Furniture (confidence=0.84)


## 8) Async usage

The SDK also provides `apredict()` for asynchronous workflows. This is useful in web servers, batch pipelines, or when making concurrent prediction calls.

In [14]:
async_request = RPTRequest(
    prediction_config=PredictionConfig(
        target_columns=[
            TargetColumn(name="COSTCENTER", task_type="classification")
        ]
    ),
    rows=[
        {"PRODUCT": "Couch",        "PRICE": 999.99, "ORDERDATE": "28-11-2025", "ID": "35",  "COSTCENTER": "[PREDICT]"},
        {"PRODUCT": "Office Chair", "PRICE": 150.80, "ORDERDATE": "02-11-2025", "ID": "44",  "COSTCENTER": "Office Furniture"},
        {"PRODUCT": "Server Rack",  "PRICE": 2200.00,"ORDERDATE": "01-11-2025", "ID": "104", "COSTCENTER": "Data Infrastructure"},
    ],
    index_column="ID",
)

response = await client.apredict(body=async_request, model_name=MODEL_NAME)

print("Status:     ", response.status)
print("Predictions:", response.predictions)

Status:      code=0 message='ok'
Predictions: [Prediction(root={'COSTCENTER': [PredictionItem(prediction='Office Furniture', confidence=0.97)], 'ID': 35})]


## 9) Pandas DataFrame workflow

In practice, data lives in CSV files, databases, or DataFrames — not hand-typed dictionaries. This section shows a realistic workflow:

1. Load a CSV into a pandas DataFrame
2. Split into **context rows** (known targets) and **query rows** (to predict)
3. Inject `"[PREDICT]"` placeholders into the query rows' target column
4. Convert to the SDK's `rows` format via `DataFrame.to_dict(orient="records")`

> **Date format note:** When using `data_schema` with `{"dtype": "date"}`, dates must be in ISO format (`YYYY-MM-DD`). Non-ISO formats like `DD-MM-YYYY` work fine when the schema is omitted (auto-inference handles them), but explicit `date` typing requires ISO.

In [15]:
import pandas as pd

df = pd.read_csv("sample_orders.csv", dtype={"ID": str})

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df

Loaded 10 rows, 7 columns


,ID,PRODUCT,PRICE,ORDERDATE,QUANTITY,COSTCENTER,DISCOUNT_RATE
0,44,Office Chair,150.80,2025-11-02,12,Office Furniture,0.08
1,104,Server Rack,2200.00,2025-11-01,2,Data Infrastructure,0.15
2,88,Laptop Stand,49.90,2025-11-03,25,Office Furniture,0.05
3,205,Standing Desk,640.00,2025-11-05,6,Office Furniture,0.10
4,306,Monitor 27 inch,289.99,2025-11-08,10,Office Furniture,0.07
5,401,Network Switch,980.00,2025-11-10,3,Data Infrastructure,0.12
6,502,Keyboard Mechanical,120.00,2025-11-12,30,Office Furniture,0.06
7,35,Couch,999.99,2025-11-28,4,Office Furniture,0.09
8,610,Cable Management Kit,25.50,2025-11-15,50,Office Furniture,0.03
9,711,UPS Battery Backup,450.00,2025-11-18,5,Data Infrastructure,0.11


Split the data into **context** (rows with known target values) and **query** (rows where we want predictions). Then replace the target column in the query rows with the `[PREDICT]` placeholder and recombine into a single DataFrame.

In [16]:
context_df = df.iloc[:-2].copy()
query_df = df.iloc[-2:].copy()

# Inject the prediction placeholder into the target column for query rows.
query_df["COSTCENTER"] = "[PREDICT]"

# Recombine: context rows first, then query rows with placeholders.
combined_df = pd.concat([context_df, query_df], ignore_index=True)

print(f"Context rows: {len(context_df)}, Query rows: {len(query_df)}")
combined_df

Context rows: 8, Query rows: 2


,ID,PRODUCT,PRICE,ORDERDATE,QUANTITY,COSTCENTER,DISCOUNT_RATE
0,44,Office Chair,150.80,2025-11-02,12,Office Furniture,0.08
1,104,Server Rack,2200.00,2025-11-01,2,Data Infrastructure,0.15
2,88,Laptop Stand,49.90,2025-11-03,25,Office Furniture,0.05
3,205,Standing Desk,640.00,2025-11-05,6,Office Furniture,0.10
4,306,Monitor 27 inch,289.99,2025-11-08,10,Office Furniture,0.07
5,401,Network Switch,980.00,2025-11-10,3,Data Infrastructure,0.12
6,502,Keyboard Mechanical,120.00,2025-11-12,30,Office Furniture,0.06
7,35,Couch,999.99,2025-11-28,4,Office Furniture,0.09
8,610,Cable Management Kit,25.50,2025-11-15,50,[PREDICT],0.03
9,711,UPS Battery Backup,450.00,2025-11-18,5,[PREDICT],0.11


Convert the DataFrame to a list of dictionaries and send the prediction request. The `to_dict(orient="records")` method maps directly to RPT-1's `rows` format.

In [17]:
rows = combined_df.to_dict(orient="records")

body = RPTRequest(
    prediction_config=PredictionConfig(
        target_columns=[
            TargetColumn(name="COSTCENTER", task_type="classification")
        ]
    ),
    rows=rows,
    index_column="ID",
    data_schema={
        "ID":            {"dtype": "string"},
        "PRODUCT":       {"dtype": "string"},
        "PRICE":         {"dtype": "numeric"},
        "ORDERDATE":     {"dtype": "date"},
        "QUANTITY":      {"dtype": "numeric"},
        "COSTCENTER":    {"dtype": "string"},
        "DISCOUNT_RATE": {"dtype": "numeric"},
    },
)

response = client.predict(body=body, model_name=MODEL_NAME)

print("Status:  ", response.status)
print("Metadata:", response.metadata)

print("\nPredictions:")
for pred in response.predictions:
    row_data = pred.root
    row_id = row_data.get("ID", "?")
    for item in row_data["COSTCENTER"]:
        print(f"  ID={row_id}: {item.prediction} (confidence={item.confidence})")

Status:   code=0 message='ok'
Metadata: num_rows=10 num_columns=7 num_predictions=2 num_query_rows=2

Predictions:
  ID=610: Office Furniture (confidence=0.99)
  ID=711: Office Furniture (confidence=0.65)


## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `401/403` error | Invalid credentials or missing scopes | Verify `AICORE_CLIENT_ID`, `AICORE_CLIENT_SECRET`, and role collections |
| `404` on predict | No running RPT-1 deployment | Create a deployment via AI Launchpad or `RPT-1_request.ipynb` |
| `422` or status code `2` | Malformed payload | Check column names, placeholder values, and `data_schema` types |
| Status code `1` (warning) | Invalid values in target column | Predictions are returned but quality may be impacted — clean your data |
| Low prediction quality | Insufficient or noisy context data | Add more representative context rows, use explicit `data_schema` and `task_type` |

### Best practices

- Use clean data with minimal missing values
- Use descriptive column names (under 100 characters)
- Keep data types consistent within each column
- Provide explicit `data_schema` and `task_type` for best results
- Maximum of **128 query rows** per request